In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine

BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
CLEAN = BASE / "data" / "cleaned"
engine = create_engine(f"sqlite:///{BASE / 'cart2insights.db'}")

order_dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"]

orders    = pd.read_sql("SELECT * FROM orders", engine, parse_dates=order_dates)
items     = pd.read_sql("SELECT * FROM items", engine)
payments  = pd.read_sql("SELECT * FROM payments", engine)
reviews   = pd.read_sql("SELECT review_id, order_id, review_score FROM reviews", engine)
customers = pd.read_sql("SELECT customer_id, customer_unique_id, customer_state, customer_city FROM customers", engine)
products  = pd.read_sql("SELECT product_id, product_category_name FROM products", engine)
sellers   = pd.read_sql("SELECT * FROM sellers", engine)
category_translation = pd.read_sql("SELECT * FROM category_translation", engine)

print("orders", orders.shape, "| items", items.shape, "| payments", payments.shape)

orders (99441, 8) | items (112650, 7) | payments (103877, 5)


In [2]:
# Total payment per order
pay_agg = payments.groupby("order_id", as_index=False).agg(
    order_value=("payment_value", "sum"),
    payment_count=("payment_sequential", "count"),
    max_installments=("payment_installments", "max"),
)

# Items per order
item_agg = items.groupby("order_id", as_index=False).agg(
    items_total=("price", "sum"),
    freight_total=("freight_value", "sum"),
    item_count=("order_item_id", "count"),
)

# Main payment type (the payment with the highest value)
main_pay = (payments.sort_values("payment_value", ascending=False)
                    .drop_duplicates("order_id")[["order_id", "payment_type"]]
                    .rename(columns={"payment_type": "main_payment_type"}))

order_features = (
    orders.merge(customers[["customer_id", "customer_unique_id", "customer_state"]],
                 on="customer_id", how="left")
          .merge(pay_agg, on="order_id", how="left")
          .merge(item_agg, on="order_id", how="left")
          .merge(main_pay, on="order_id", how="left")
          .merge(reviews.drop_duplicates("order_id")[["order_id", "review_score"]],
                 on="order_id", how="left")
)

# Delivery features (only meaningful when the order was actually delivered)
secs_per_day = 86400
order_features["delivery_days"] = (
    (order_features["order_delivered_customer_date"] - order_features["order_purchase_timestamp"])
    .dt.total_seconds() / secs_per_day
).round(1)

order_features["delivery_delay_days"] = (
    (order_features["order_delivered_customer_date"] - order_features["order_estimated_delivery_date"])
    .dt.total_seconds() / secs_per_day
).round(1)

# 1 = delayed, 0 = on time, empty = not delivered yet
order_features["is_delayed"] = np.where(
    order_features["delivery_delay_days"].isna(), np.nan,
    (order_features["delivery_delay_days"] > 0).astype(float)
)

order_features["delivery_status"] = np.select(
    [order_features["is_delayed"] == 1, order_features["is_delayed"] == 0],
    ["delayed", "on_time"], default="not_delivered"
)

# Time features for trend analysis
order_features["purchase_month"] = order_features["order_purchase_timestamp"].dt.to_period("M").astype(str)
order_features["purchase_year"] = order_features["order_purchase_timestamp"].dt.year
order_features["purchase_weekday"] = order_features["order_purchase_timestamp"].dt.day_name()

print(order_features.shape)
order_features[["order_id", "order_value", "delivery_days", "delivery_delay_days",
                "delivery_status", "review_score"]].head(10)

(99441, 25)


,order_id,order_value,delivery_days,delivery_delay_days,delivery_status,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,38.71,8.4,-7.1,on_time,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,141.46,13.8,-5.4,on_time,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,179.12,9.4,-17.2,on_time,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,72.20,13.2,-13.0,on_time,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,28.62,2.9,-9.2,on_time,5.0
5,a4591c265e18cb1dcee52889e2d8acc3,175.26,16.5,-5.5,on_time,4.0
6,136cce7faa42fdb2cefd53fdc79a6098,65.95,NaN,NaN,not_delivered,2.0
7,6514b8ad8028c9f2cc2374ded245783f,75.16,10.0,-11.5,on_time,5.0
8,76c6e866289321a7c93b82b54852dc33,35.95,9.8,-31.4,on_time,1.0
9,e69bfb5eb88e0ed6a785585b27e16dbf,169.76,18.2,-6.3,on_time,5.0


In [3]:
valid_orders = order_features[
    ~order_features["order_status"].isin(["canceled", "unavailable"])
    & order_features["order_value"].notna()
]

customer_features = valid_orders.groupby("customer_unique_id", as_index=False).agg(
    customer_order_count=("order_id", "nunique"),
    customer_total_spending=("order_value", "sum"),
    customer_avg_order_value=("order_value", "mean"),
    first_order_date=("order_purchase_timestamp", "min"),
    last_order_date=("order_purchase_timestamp", "max"),
)

customer_features["is_repeat_customer"] = (customer_features["customer_order_count"] > 1).astype(int)

# Add the customer's state
state_map = customers.drop_duplicates("customer_unique_id")[["customer_unique_id", "customer_state"]]
customer_features = customer_features.merge(state_map, on="customer_unique_id", how="left")

customer_features["customer_total_spending"] = customer_features["customer_total_spending"].round(2)
customer_features["customer_avg_order_value"] = customer_features["customer_avg_order_value"].round(2)

print(customer_features.shape)
print("Repeat customers:", customer_features["is_repeat_customer"].sum())
customer_features.head()

(94989, 8)
Repeat customers: 2888


,customer_unique_id,customer_order_count,customer_total_spending,customer_avg_order_value,first_order_date,last_order_date,is_repeat_customer,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,2018-05-10 10:56:27,2018-05-10 10:56:27,0,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,2018-05-07 11:11:27,2018-05-07 11:11:27,0,SP
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,2017-03-10 21:05:03,2017-03-10 21:05:03,0,SC
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,2017-10-12 20:29:41,2017-10-12 20:29:41,0,PA
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,2017-11-14 19:45:42,2017-11-14 19:45:42,0,SP


In [4]:
seller_base = items.groupby("seller_id", as_index=False).agg(
    seller_revenue=("price", "sum"),
    seller_order_count=("order_id", "nunique"),
    seller_items_sold=("order_item_id", "count"),
)

# Seller rating and delivery performance, via the orders they shipped
seller_orders = items[["order_id", "seller_id"]].drop_duplicates().merge(
    order_features[["order_id", "review_score", "delivery_days", "is_delayed"]],
    on="order_id", how="left"
)

seller_perf = seller_orders.groupby("seller_id", as_index=False).agg(
    seller_avg_review=("review_score", "mean"),
    seller_avg_delivery_days=("delivery_days", "mean"),
    seller_delay_rate=("is_delayed", "mean"),
)

seller_features = (
    seller_base.merge(seller_perf, on="seller_id", how="left")
               .merge(sellers[["seller_id", "seller_city", "seller_state"]], on="seller_id", how="left")
)

for col in ["seller_revenue", "seller_avg_review", "seller_avg_delivery_days", "seller_delay_rate"]:
    seller_features[col] = seller_features[col].round(2)

print(seller_features.shape)
seller_features.sort_values("seller_revenue", ascending=False).head()

(3095, 9)


,seller_id,seller_revenue,seller_order_count,seller_items_sold,seller_avg_review,seller_avg_delivery_days,seller_delay_rate,seller_city,seller_state
857,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,1156,4.13,15.06,0.11,guariba,SP
1013,53243585a1d6dc2643021fd1853d8905,222776.05,358,410,4.13,13.43,0.04,lauro de freitas,BA
881,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,1987,3.83,14.49,0.11,ibitinga,SP
3024,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,586,4.34,13.35,0.10,sumare,SP
1535,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,1364,3.49,22.34,0.10,itaquaquecetuba,SP


In [5]:
item_features = (
    items.merge(products, on="product_id", how="left")
         .merge(category_translation, on="product_category_name", how="left")
)
item_features["category"] = item_features["product_category_name_english"].fillna("unknown")
item_features["item_total"] = item_features["price"] + item_features["freight_value"]

print(item_features.shape)
item_features[["order_id", "category", "price", "freight_value", "item_total"]].head()

(112650, 11)


,order_id,category,price,freight_value,item_total
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,perfumery,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,199.90,18.14,218.04


In [6]:
feature_tables = {
    "order_features": order_features,
    "customer_features": customer_features,
    "seller_features": seller_features,
    "item_features": item_features,
}

for name, df in feature_tables.items():
    df.to_sql(name, engine, if_exists="replace", index=False, chunksize=5000)
    df.to_csv(CLEAN / f"{name}.csv", index=False)
    print("saved", name, df.shape)

saved order_features (99441, 25)
saved customer_features (94989, 8)
saved seller_features (3095, 9)
saved item_features (112650, 11)


In [7]:
for name in feature_tables:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {name}", engine)["n"][0]
    print(f"{name:20s} rows in database: {n}")

print("\nDelivery status counts:")
print(order_features["delivery_status"].value_counts())

print("\nAverage review score by delivery status:")
print(order_features.groupby("delivery_status")["review_score"].mean().round(2))

order_features       rows in database: 99441
customer_features    rows in database: 94989
seller_features      rows in database: 3095
item_features        rows in database: 112650

Delivery status counts:
delivery_status
on_time          88710
delayed           7766
not_delivered     2965
Name: count, dtype: int64

Average review score by delivery status:
delivery_status
delayed          2.55
not_delivered    1.74
on_time          4.30
Name: review_score, dtype: float64


## Features created
- **Order level:** order_value, items_total, freight_total, item_count, delivery_days, delivery_delay_days, is_delayed, delivery_status, purchase_month
- **Customer level:** customer_order_count, customer_total_spending, customer_avg_order_value, is_repeat_customer
- **Seller level:** seller_revenue, seller_order_count, seller_avg_review, seller_avg_delivery_days, seller_delay_rate
- **Item level:** category (English), item_total
